# LLM GMC — Name That LLM baseline

This competition-linked notebook computes a reproducible local validation score. It does not submit predictions to the leaderboard.

Baseline: word + character TF-IDF features with class-balanced LinearSVC. If `prompt_id` is present, validation is grouped by prompt to reduce prompt leakage.

In [ ]:
from pathlib import Path
import json
import time
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

DATA_DIR = Path('/kaggle/input/competitions/name-that-llm')
print('Dataset directory exists:', DATA_DIR.exists())
print('Dataset files:', sorted(p.name for p in DATA_DIR.glob('*')) if DATA_DIR.exists() else [])
assert (DATA_DIR / 'train.csv').exists(), f'train.csv not found under {DATA_DIR}'
train = pd.read_csv(DATA_DIR / 'train.csv')
print('Train shape:', train.shape)
print('Columns:', train.columns.tolist())
assert {'text', 'model'}.issubset(train.columns), 'Expected text and model columns'
train['text'] = train['text'].fillna('').astype(str)
train['model'] = train['model'].astype(str)
print('Model classes:', train['model'].nunique())
display(train['model'].value_counts())

In [ ]:
if 'prompt_id' in train.columns and train['prompt_id'].notna().any():
    groups = train['prompt_id'].fillna('__missing_prompt_id__').astype(str)
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    train_idx, valid_idx = next(splitter.split(train, groups=groups))
    split_method = 'GroupShuffleSplit by prompt_id'
else:
    train_idx, valid_idx = train_test_split(range(len(train)), test_size=0.20, random_state=42, stratify=train['model'])
    split_method = 'stratified random split (prompt_id unavailable)'

train_part = train.iloc[list(train_idx)].copy()
valid_part = train.iloc[list(valid_idx)].copy()
print('Validation method:', split_method)
print('Training rows:', len(train_part))
print('Validation rows:', len(valid_part))
print('Validation classes:', valid_part['model'].nunique())

In [ ]:
import re
import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import StandardScaler

# Fixed-fold TF-IDF ablation. Every configuration uses exactly the same prompt groups.
texts = train['text'].fillna('').astype(str).to_numpy()
labels = train['model'].astype(str).to_numpy()
groups = train['prompt_id'].fillna('__missing_prompt_id__').astype(str).to_numpy()
style = []
for text in texts:
    words = re.findall(r"\b[\w']+\b", text.lower())
    n_words, n_chars = max(len(words),1), max(len(text),1)
    lengths = np.asarray([len(w) for w in words], dtype=float)
    style.append([
        np.log1p(len(text)), np.log1p(len(words)), np.log1p(max(len(re.findall(r'[.!?]+', text)),1)),
        lengths.mean() if len(lengths) else 0., lengths.std() if len(lengths) else 0.,
        len(set(words))/n_words, sum(c.isdigit() for c in text)/n_chars,
        sum(c.isupper() for c in text)/n_chars, sum(c.isspace() for c in text)/n_chars,
        text.count('\\n')/n_chars, text.count('\\n\\n')/n_chars,
        *[text.count(c)/n_chars for c in '.,;:!?()[]{}"\'/-'],
        text.count('#')/n_chars, text.count('*')/n_chars, text.count('```')/n_chars,
    ])
style = np.asarray(style, dtype=np.float32)

configs = {
    'word_1_3': [('word','word',(1,3))],
    'char_2_5': [('char','char',(2,5))],
    'char_3_6': [('char','char',(3,6))],
    'char_wb_2_5': [('char_wb','char_wb',(2,5))],
    'word_1_3_char_2_5': [('word','word',(1,3)),('char','char',(2,5))],
    'word_1_3_char_wb_2_5': [('word','word',(1,3)),('char_wb','char_wb',(2,5))],
    'word_char_style': [('word','word',(1,3)),('char','char',(2,5)),'STYLE'],
}

def make_union(spec):
    pieces=[]
    for name, analyzer, rng in spec:
        pieces.append((name, TfidfVectorizer(analyzer=analyzer, ngram_range=rng, min_df=2, sublinear_tf=True, max_features=120000)))
    return FeatureUnion(pieces)

fold_indices = list(GroupKFold(n_splits=5).split(texts, labels, groups=groups))
all_results = {}
for config_name, spec in configs.items():
    fold_scores=[]
    for fold, (tr, va) in enumerate(fold_indices,1):
        text_spec=[x for x in spec if x != 'STYLE']
        tf=make_union(text_spec)
        Xtr_text=tf.fit_transform(texts[tr]); Xva_text=tf.transform(texts[va])
        if 'STYLE' in spec:
            scaler=StandardScaler()
            Xtr=hstack([Xtr_text, csr_matrix(scaler.fit_transform(style[tr]))],format='csr')
            Xva=hstack([Xva_text, csr_matrix(scaler.transform(style[va]))],format='csr')
        else:
            Xtr, Xva = Xtr_text, Xva_text
        model=LinearSVC(C=1.0,class_weight='balanced',max_iter=1200)
        model.fit(Xtr,labels[tr])
        score=float(balanced_accuracy_score(labels[va],model.predict(Xva)))
        fold_scores.append(score)
        print(f'{config_name} fold {fold}: {score:.4f}')
    all_results[config_name]={
        'mean_balanced_accuracy':float(np.mean(fold_scores)),
        'std_balanced_accuracy':float(np.std(fold_scores,ddof=1)),
        'fold_scores':fold_scores,
    }
    print(f'CONFIG {config_name}: mean={np.mean(fold_scores):.4f}')

ranked=sorted(all_results.items(),key=lambda kv:kv[1]['mean_balanced_accuracy'],reverse=True)
summary={'experiment':'fixed-fold TF-IDF and stylometry ablation','split_method':'5-fold GroupKFold by prompt_id','configs':all_results,'ranking':[{'config':k,**v} for k,v in ranked]}
print('\\nSUMMARY:',json.dumps(summary,indent=2))
Path('/kaggle/working/ablation_results.json').write_text(json.dumps(summary,indent=2))
print('Saved:', '/kaggle/working/ablation_results.json')


## Fixed-fold TF-IDF ablation

All configurations use identical prompt-grouped folds and class-balanced LinearSVC.
